<font color='red'><b>**WARNING**</b></font> <br/>
어떠한 사유로도 임의로 복사, 촬영, 녹음, 복제, 보관, 전송하거나 허가 받지 않은 저장매체를 이용한 보관, 제3자에게 누설, 공개 또는 사용하는 등의 무단 사용 및 불법 배포 시 법적 조치를 받을 수 있습니다. <br/>

<div style="text-align: right; color: #7f8c8d; font-size: 0.9em; margin-top: 20px;">
📝 Author: 박사홍 (Sahong Pak)</br>
📧 Contact: sahong.pak@gmail.com</br>
📌 Version: v2.0</br>
📅 Last Updated: 2026-03-12</br>
</div>

</br>

# 학습 내용
>이번 장에서는 <strong>ResNet-50 비교(ResNet-50 Comparison)</strong>에 대해 학습합니다.</br></br>
>ResNet-50으로 추론 파이프라인을 구성하고 CLIP과의 분류 결과를 학습해봅시다.

</br>

# ResNet-50 추론 (Inference)
> <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">ImageNet 사전학습된 ResNet-50</mark>으로 이미지 분류를 수행합니다.

신경망을 단순히 깊게 쌓으면 성능이 오히려 저하되는 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">기울기 소실(Vanishing Gradient)</mark> 문제가 발생합니다. 역전파 시 기울기가 층을 거치며 점점 작아져 앞쪽 층의 가중치가 거의 업데이트되지 않기 때문입니다. ResNet은 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">잔차 연결(Skip Connection)</mark>을 도입 하여 이 문제를 해결합니다. 입력을 변환하지 않고 출력에 직접 더하는 단축 경로(shortcut)를 통해 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">기울기가 깊은 층까지 직접 전달</mark>되므로, 50층 이상의 매우 깊은 네트워크도 안정적으로 학습할 수 있 습니다.</br>이 장에서는 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">CNN</mark>(합성곱 연산으로 이미지의 지역적 패턴을 계층적으로 추출하는 구조), <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">풀링</mark>(특성 맵의 공간 해상도를 줄여 계산량과 과적합을 감소시키는 연산), <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">배치 정규화</mark>(각 층의 출력을 정규화하여 학습을 안정화), 그리고 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">Skip Connection</mark>(입력을 한 개 이상의 층을 건너뛰어 출력에 직접 더하는 구조) 개념을 활용합니다.

In [1]:
# TODO 1: 사전학습된 ResNet-50 모델을 로드하고, 모델을 평가 모드로 설정한 뒤, 해당 가중치에 맞는 전처리를 설정해봅시다.

import torchvision.models as models
from torchvision import transforms
import torch

# 사전학습 모델 로드
weights = models.ResNet50_Weights.IMAGENET1K_V2
model = models.resnet50(weights=weights)
model.eval()

# ImageNet 전처리
preprocess = weights.transforms()
print(f"모델 파라미터: {sum(p.numel() for p in model.parameters()):,}")
print(f"전처리: {preprocess}")

Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to C:\Users\SSAFY/.cache\torch\hub\checkpoints\resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:05<00:00, 18.1MB/s]


모델 파라미터: 25,557,032
전처리: ImageClassification(
    crop_size=[224]
    resize_size=[232]
    mean=[0.485, 0.456, 0.406]
    std=[0.229, 0.224, 0.225]
    interpolation=InterpolationMode.BILINEAR
)


</br>

## 추론 파이프라인

In [2]:
# TODO 2: 이미지를 전처리하고 배치 차원을 추가한 뒤, 추론 모드에서 모델을 실행하여 Top-5 예측 결과를 출력해봅시다.

from PIL import Image

# 이미지 로드 및 전처리
image = Image.open("test.jpg")
input_tensor = preprocess(image).unsqueeze(0)  # 배치 차원 추가
print(f"입력 텐서: {input_tensor.shape}")

# 추론
with torch.no_grad():
    output = model(input_tensor)
    probabilities = torch.softmax(output, dim=1)

# Top-5 예측
top5_probs, top5_indices = probabilities.topk(5)
categories = weights.meta["categories"]

print(f"\nTop-5 예측:")
for i in range(5):
    idx = top5_indices[0][i].item()
    prob = top5_probs[0][i].item()
    print(f"  {categories[idx]}: {prob:.2%}")

FileNotFoundError: [Errno 2] No such file or directory: 'test.jpg'

<table style="width:100%">
  <thead>
    <tr>
      <th style="text-align:center">순위</th>
      <th style="text-align:center">카테고리</th>
      <th style="text-align:center">확률</th>
    </tr>
  </thead>
  <tbody>
    <tr><td style="text-align:center">1</td><td style="text-align:center"><mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">tabby</mark></td><td style="text-align:center">42.58%</td></tr>
    <tr><td style="text-align:center">2</td><td style="text-align:center">tiger cat</td><td style="text-align:center">35.21%</td></tr>
    <tr><td style="text-align:center">3</td><td style="text-align:center">Egyptian cat</td><td style="text-align:center">12.34%</td></tr>
    <tr><td style="text-align:center">4</td><td style="text-align:center">lynx</td><td style="text-align:center">3.87%</td></tr>
    <tr><td style="text-align:center">5</td><td style="text-align:center">Persian cat</td><td style="text-align:center">1.56%</td></tr>
  </tbody>
</table>

</br>

## ResNet-50 vs CLIP 비교

<table style="width:100%">
  <thead>
    <tr>
      <th style="text-align:center">비교 항목</th>
      <th>ResNet-50</th>
      <th>CLIP</th>
    </tr>
  </thead>
  <tbody>
    <tr><td style="text-align:center">분류 방식</td><td><mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">고정 1000 클래스</mark></td><td>자유 텍스트 매칭</td></tr>
    <tr><td style="text-align:center">새 클래스</td><td>재학습 필요</td><td>Zero-shot 가능</td></tr>
    <tr><td style="text-align:center">입력</td><td>이미지만</td><td>이미지 + 텍스트</td></tr>
    <tr><td style="text-align:center">강점</td><td>ImageNet 최적화</td><td>유연한 분류</td></tr>
  </tbody>
</table>

💡weights.transforms()
> `weights.transforms()`는 해당 모델 학습 시 사용된 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">정확한 전처리</mark>를 반환합니다.</br>
> 수동으로 Normalize 값을 입력하는 것보다 안전합니다.

💡어떤 모델을 선택할까?
> 고정된 카테고리 분류: <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">ResNet-50</mark> (빠르고 정확)</br>
> 유연한 분류/검색: <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">CLIP</mark> (새 카테고리 추가 가능, Zero-shot)